In [15]:
import xarray as xr
from glob import glob
import pandas as pd
import numpy as np
from tqdm import tqdm

In [16]:
def braced_glob(path):
    l = []
    for x in braceexpand(path):
        l.extend(glob(x))
            
    return l

In [17]:
sim = 'UBI'

In [19]:
input_dir  = f'/home/vdemeyer/projects/rrg-gachon/vdemeyer/{sim}/PSL'

for year in tqdm(range(2010, 2101)):

    input_files = sorted(glob(f'{input_dir}/psl_{sim.lower()}_{year}*.nc'))
    if not input_files:
        continue

    ds = xr.open_mfdataset(input_files)
    ds['time'] = ds['time'].dt.round('h')

    # Définition des bornes temporelles attendues
    if sim in ['UBG', 'UBH', 'UBI'] and year == 2015:
        start, end = f"{year}-01-01 01:00:00", f"{year}-12-31 23:00:00"
    elif sim == 'UBI' and year == 2098:
        start, end = f"{year}-01-01 00:00:00", f"{year}-04-30 23:00:00"
    else:
        start, end = f"{year}-01-01 00:00:00", f"{year}-12-31 23:00:00"

    expected_time = pd.date_range(start=start, end=end, freq="h").to_numpy()
    time_ds = ds['time'].to_numpy()

    # --- Vérifications temporelles ---
    if np.array_equal(time_ds, expected_time):
        continue

    print(f"\n⚠️ {year}: time coordinate mismatch detected!")

    # Tri si besoin
    if not np.all(time_ds[:-1] <= time_ds[1:]):
        print("   ↳ Time is not sorted. Sorting it now.")
        ds = ds.sortby("time")
        time_ds = ds["time"].to_numpy()

    # Doublons et heures manquantes
    unique_times, counts = np.unique(time_ds, return_counts=True)
    duplicated = unique_times[counts > 1]
    missing = set(expected_time) - set(unique_times)

    if duplicated.size > 0:
        print(f"   ❌ {len(duplicated)} duplicated timestamps (first 3 shown):")
        for t in duplicated[:3]:
            print(f"      {t}")

    if missing:
        print(f"   ❌ {len(missing)} missing timestamps (first 3 shown):")
        for t in sorted(list(missing))[:3]:
            print(f"      {t}")

    # Comparaison directe sur les indices communs
    min_len = min(len(time_ds), len(expected_time))
    diff_indices = np.where(time_ds[:min_len] != expected_time[:min_len])[0]

    if len(diff_indices) > 0:
        print(f"   ❌ {len(diff_indices)} differing timestamps (first 3 shown):")
        for i in diff_indices[:3]:
            print(f"      Index {i}: ds = {time_ds[i]}, expected = {expected_time[i]}")

    print(f"   → ds['time'] length = {len(time_ds)}, expected = {len(expected_time)}")

 23%|██▎       | 21/91 [00:35<02:33,  2.19s/it]


⚠️ 2030: time coordinate mismatch detected!
   ❌ 1 missing timestamps (first 3 shown):
      2030-04-01T00:00:00.000000000
   ❌ 6599 differing timestamps (first 3 shown):
      Index 2160: ds = 2030-04-01T01:00:00.000000000, expected = 2030-04-01T00:00:00.000000000
      Index 2161: ds = 2030-04-01T02:00:00.000000000, expected = 2030-04-01T01:00:00.000000000
      Index 2162: ds = 2030-04-01T03:00:00.000000000, expected = 2030-04-01T02:00:00.000000000
   → ds['time'] length = 8759, expected = 8760


 29%|██▊       | 26/91 [00:46<02:26,  2.25s/it]


⚠️ 2035: time coordinate mismatch detected!
   ❌ 1 missing timestamps (first 3 shown):
      2035-05-01T00:00:00.000000000
   ❌ 5879 differing timestamps (first 3 shown):
      Index 2880: ds = 2035-05-01T01:00:00.000000000, expected = 2035-05-01T00:00:00.000000000
      Index 2881: ds = 2035-05-01T02:00:00.000000000, expected = 2035-05-01T01:00:00.000000000
      Index 2882: ds = 2035-05-01T03:00:00.000000000, expected = 2035-05-01T02:00:00.000000000
   → ds['time'] length = 8759, expected = 8760


 34%|███▍      | 31/91 [00:57<02:14,  2.24s/it]


⚠️ 2040: time coordinate mismatch detected!
   ❌ 1 missing timestamps (first 3 shown):
      2040-06-01T00:00:00.000000000
   ❌ 5135 differing timestamps (first 3 shown):
      Index 3648: ds = 2040-06-01T01:00:00.000000000, expected = 2040-06-01T00:00:00.000000000
      Index 3649: ds = 2040-06-01T02:00:00.000000000, expected = 2040-06-01T01:00:00.000000000
      Index 3650: ds = 2040-06-01T03:00:00.000000000, expected = 2040-06-01T02:00:00.000000000
   → ds['time'] length = 8783, expected = 8784


100%|██████████| 91/91 [04:46<00:00,  3.15s/it]


In [21]:
input_dir  = f'/home/vdemeyer/projects/rrg-gachon/vdemeyer/{sim}/UAS'

for year in tqdm(range(2010, 2101)):

    input_files = sorted(glob(f'{input_dir}/uas_{sim.lower()}_{year}*.nc'))
    if not input_files:
        continue

    ds = xr.open_mfdataset(input_files)
    ds['time'] = ds['time'].dt.round('h')

    # Définition des bornes temporelles attendues
    if sim in ['UBG', 'UBH', 'UBI'] and year == 2015:
        start, end = f"{year}-01-01 01:00:00", f"{year}-12-31 23:00:00"
    elif sim == 'UBI' and year == 2098:
        start, end = f"{year}-01-01 00:00:00", f"{year}-04-30 23:00:00"
    else:
        start, end = f"{year}-01-01 00:00:00", f"{year}-12-31 23:00:00"

    expected_time = pd.date_range(start=start, end=end, freq="h").to_numpy()
    time_ds = ds['time'].to_numpy()

    # --- Vérifications temporelles ---
    if np.array_equal(time_ds, expected_time):
        continue

    print(f"\n⚠️ {year}: time coordinate mismatch detected!")

    # Tri si besoin
    if not np.all(time_ds[:-1] <= time_ds[1:]):
        print("   ↳ Time is not sorted. Sorting it now.")
        ds = ds.sortby("time")
        time_ds = ds["time"].to_numpy()

    # Doublons et heures manquantes
    unique_times, counts = np.unique(time_ds, return_counts=True)
    duplicated = unique_times[counts > 1]
    missing = set(expected_time) - set(unique_times)

    if duplicated.size > 0:
        print(f"   ❌ {len(duplicated)} duplicated timestamps (first 3 shown):")
        for t in duplicated[:3]:
            print(f"      {t}")

    if missing:
        print(f"   ❌ {len(missing)} missing timestamps (first 3 shown):")
        for t in sorted(list(missing))[:3]:
            print(f"      {t}")

    # Comparaison directe sur les indices communs
    min_len = min(len(time_ds), len(expected_time))
    diff_indices = np.where(time_ds[:min_len] != expected_time[:min_len])[0]

    if len(diff_indices) > 0:
        print(f"   ❌ {len(diff_indices)} differing timestamps (first 3 shown):")
        for i in diff_indices[:3]:
            print(f"      Index {i}: ds = {time_ds[i]}, expected = {expected_time[i]}")

    print(f"   → ds['time'] length = {len(time_ds)}, expected = {len(expected_time)}")

 23%|██▎       | 21/91 [01:12<05:18,  4.55s/it]


⚠️ 2030: time coordinate mismatch detected!
   ❌ 1 missing timestamps (first 3 shown):
      2030-04-01T00:00:00.000000000
   ❌ 6599 differing timestamps (first 3 shown):
      Index 2160: ds = 2030-04-01T01:00:00.000000000, expected = 2030-04-01T00:00:00.000000000
      Index 2161: ds = 2030-04-01T02:00:00.000000000, expected = 2030-04-01T01:00:00.000000000
      Index 2162: ds = 2030-04-01T03:00:00.000000000, expected = 2030-04-01T02:00:00.000000000
   → ds['time'] length = 8759, expected = 8760


 29%|██▊       | 26/91 [01:36<05:01,  4.64s/it]


⚠️ 2035: time coordinate mismatch detected!
   ❌ 1 missing timestamps (first 3 shown):
      2035-05-01T00:00:00.000000000
   ❌ 5879 differing timestamps (first 3 shown):
      Index 2880: ds = 2035-05-01T01:00:00.000000000, expected = 2035-05-01T00:00:00.000000000
      Index 2881: ds = 2035-05-01T02:00:00.000000000, expected = 2035-05-01T01:00:00.000000000
      Index 2882: ds = 2035-05-01T03:00:00.000000000, expected = 2035-05-01T02:00:00.000000000
   → ds['time'] length = 8759, expected = 8760


 34%|███▍      | 31/91 [01:59<04:34,  4.57s/it]


⚠️ 2040: time coordinate mismatch detected!
   ❌ 1 missing timestamps (first 3 shown):
      2040-06-01T00:00:00.000000000
   ❌ 5135 differing timestamps (first 3 shown):
      Index 3648: ds = 2040-06-01T01:00:00.000000000, expected = 2040-06-01T00:00:00.000000000
      Index 3649: ds = 2040-06-01T02:00:00.000000000, expected = 2040-06-01T01:00:00.000000000
      Index 3650: ds = 2040-06-01T03:00:00.000000000, expected = 2040-06-01T02:00:00.000000000
   → ds['time'] length = 8783, expected = 8784


 54%|█████▍    | 49/91 [03:20<03:05,  4.43s/it]


⚠️ 2058: time coordinate mismatch detected!
   ❌ 744 missing timestamps (first 3 shown):
      2058-12-01T00:00:00.000000000
      2058-12-01T01:00:00.000000000
      2058-12-01T02:00:00.000000000
   → ds['time'] length = 8016, expected = 8760


100%|██████████| 91/91 [06:22<00:00,  4.21s/it]
